**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 21: LoRA vs FFT 실전 비교 실습

## 🎯 LoRA vs FFT 실전 비교 실습

이전 세션에서 이론적 비교를 했다면, 이번 세션에서는 **동일한 데이터**로 LoRA와 FFT를 실제 학습하고 **추론 품질까지 비교**합니다.

### 실험 설계

| 항목 | LoRA | FFT |
|------|------|-----|
| 모델 | Qwen2.5-1.5B (4bit) | Qwen2.5-1.5B (FP16) |
| 데이터 | Alpaca 한국어 샘플 (동일) | Alpaca 한국어 샘플 (동일) |
| 에포크 | 3 | 3 |
| batch_size | 1 | 1 |
| 평가 | 동일 프롬프트 5개 | 동일 프롬프트 5개 |

### 학습 목표

- ✅ 동일 조건에서 LoRA와 FFT의 실제 성능 차이 확인
- ✅ 학습 곡선(loss) 비교
- ✅ 추론 품질 비교
- ✅ 종합적인 비용-효과 분석

## 1️⃣ 공통 설정 (데이터, 평가 기준)

In [ ]:
# 필수 라이브러리
import torch
import gc
import os
import json
import time
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print("✅ 라이브러리 임포트 완료")
print(f"CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

In [ ]:
# GPU 메모리 모니터링 함수
def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")

def get_gpu_memory_gb():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**3
    return 0

# 공통 설정
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 1024

# 평가용 테스트 프롬프트
TEST_PROMPTS = [
    "다음 문장을 영어로 번역하세요.\n\n인공지능은 우리 생활을 크게 변화시키고 있습니다.",
    "LoRA 파인튜닝의 장점을 3가지 설명하세요.",
    "다음 Python 코드의 버그를 찾아주세요.\n\ndef add(a, b):\n    return a - b",
    "트랜스포머 아키텍처에서 Attention 메커니즘을 쉽게 설명하세요.",
    "RTX 4060으로 LLM 학습 시 메모리 최적화 방법을 알려주세요."
]

print(f"📌 모델: {MODEL_NAME}")
print(f"📌 테스트 프롬프트 수: {len(TEST_PROMPTS)}")
print_gpu_memory("시작")

In [ ]:
# 공통 데이터 로드 및 전처리
data_path = "../data/samples/alpaca_ko_sample.json"

with open(data_path, "r", encoding="utf-8") as f:
    alpaca_data = json.load(f)

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Alpaca → Chat Template 변환
def format_alpaca_to_text(sample):
    instruction = sample["instruction"]
    input_text = sample.get("input", "")
    output_text = sample["output"]
    
    user_content = f"{instruction}\n\n{input_text}" if input_text else instruction
    messages = [
        {"role": "system", "content": "당신은 유용한 AI 어시스턴트입니다."},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output_text}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

formatted_texts = [format_alpaca_to_text(s) for s in alpaca_data]
dataset = Dataset.from_dict({"text": formatted_texts})

print(f"✅ 데이터 준비 완료: {len(dataset)}개 샘플")

# 학습 결과 저장용
results = {"lora": {}, "fft": {}}

In [ ]:
# 추론 함수
def generate_responses(model, tokenizer, prompts, max_new_tokens=150):
    """모델로 응답을 생성하는 공통 함수"""
    model.eval()
    responses = []
    for prompt in prompts:
        messages = [
            {"role": "system", "content": "당신은 유용한 AI 어시스턴트입니다."},
            {"role": "user", "content": prompt}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, temperature=1.0
            )
        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        responses.append(response)
    return responses

print("✅ 공통 함수 정의 완료")

## 2️⃣ LoRA 학습 실행 (r=16)

In [ ]:
print("="*60)
print("🔵 LoRA 학습 시작")
print("="*60)

# 4bit 양자화 + LoRA 모델 로드
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

lora_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map="auto", trust_remote_code=True
)
lora_model.gradient_checkpointing_enable()

# 학습 전 응답 저장
print("\n📋 학습 전 응답 수집 중...")
before_responses = generate_responses(lora_model, tokenizer, TEST_PROMPTS)

# LoRA 적용
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM
)
lora_model = get_peft_model(lora_model, lora_config)
lora_model.print_trainable_parameters()
print_gpu_memory("LoRA 모델 준비 완료")

In [ ]:
# LoRA SFTTrainer 학습
lora_sft_config = SFTConfig(
    output_dir="./output/comparison_lora",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    save_strategy="epoch",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    report_to="none",
    gradient_checkpointing=True,
)

lora_trainer = SFTTrainer(
    model=lora_model, args=lora_sft_config,
    train_dataset=dataset, processing_class=tokenizer,
)

print("🚀 LoRA 학습 시작!")
lora_start = time.time()
lora_result = lora_trainer.train()
lora_time = time.time() - lora_start

# 결과 저장
results["lora"]["time"] = lora_time
results["lora"]["loss"] = lora_result.training_loss
results["lora"]["memory"] = get_gpu_memory_gb()
results["lora"]["log_history"] = lora_trainer.state.log_history

print(f"\n✅ LoRA 학습 완료!")
print(f"📌 소요 시간: {lora_time:.1f}초")
print(f"📌 Final Loss: {lora_result.training_loss:.4f}")
print_gpu_memory("LoRA 학습 완료")

In [ ]:
# LoRA 학습 후 응답
print("\n📋 LoRA 학습 후 응답 수집 중...")
lora_after_responses = generate_responses(lora_model, tokenizer, TEST_PROMPTS)
results["lora"]["responses"] = lora_after_responses

# LoRA 어댑터 크기
lora_save_path = "./output/comparison_lora/lora_adapter"
lora_model.save_pretrained(lora_save_path)
lora_size = sum(
    os.path.getsize(os.path.join(root, f))
    for root, dirs, files in os.walk(lora_save_path)
    for f in files
)
results["lora"]["save_size"] = lora_size
print(f"📌 LoRA 어댑터 크기: {lora_size/1024/1024:.1f} MB")

# 메모리 해제
del lora_model, lora_trainer
gc.collect()
torch.cuda.empty_cache()
print("✅ LoRA 메모리 해제 완료")

## 3️⃣ FFT 학습 실행 (1.5B 모델)

⚠️ RTX 4060에서 FFT는 메모리가 빠듯합니다. OOM 발생 시 코드가 에러를 잡아줍니다.

In [ ]:
print("="*60)
print("🟠 FFT 학습 시작")
print("="*60)

try:
    # FP16 모델 로드 (양자화 없음)
    fft_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16,
        device_map="auto", trust_remote_code=True
    )
    fft_model.gradient_checkpointing_enable()
    
    # 학습 전 응답 (비교 기준으로 동일 모델)
    print_gpu_memory("FFT 모델 로드 후")
    
    # SFTTrainer 학습
    fft_sft_config = SFTConfig(
        output_dir="./output/comparison_fft",
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-5,  # FFT는 더 작은 lr 사용
        fp16=True,
        logging_steps=1,
        save_strategy="epoch",
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        report_to="none",
        gradient_checkpointing=True,
    )
    
    fft_trainer = SFTTrainer(
        model=fft_model, args=fft_sft_config,
        train_dataset=dataset, processing_class=tokenizer,
    )
    
    print("🚀 FFT 학습 시작!")
    fft_start = time.time()
    fft_result = fft_trainer.train()
    fft_time = time.time() - fft_start
    
    results["fft"]["time"] = fft_time
    results["fft"]["loss"] = fft_result.training_loss
    results["fft"]["memory"] = get_gpu_memory_gb()
    results["fft"]["log_history"] = fft_trainer.state.log_history
    
    print(f"\n✅ FFT 학습 완료!")
    print(f"📌 소요 시간: {fft_time:.1f}초")
    print(f"📌 Final Loss: {fft_result.training_loss:.4f}")
    print_gpu_memory("FFT 학습 완료")
    
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        print("\n⚠️ GPU 메모리 부족으로 FFT 학습 실패!")
        print("📌 이것이 바로 LoRA가 필요한 이유입니다.")
        print("📌 RTX 4060(8GB)에서 1.5B FFT는 메모리가 매우 빠듯합니다.")
        results["fft"]["time"] = float('inf')
        results["fft"]["loss"] = float('inf')
        results["fft"]["memory"] = 8.0
        gc.collect()
        torch.cuda.empty_cache()
    else:
        raise e

In [ ]:
# FFT 학습 후 응답 (학습 성공 시)
if results["fft"]["loss"] != float('inf'):
    print("📋 FFT 학습 후 응답 수집 중...")
    fft_after_responses = generate_responses(fft_model, tokenizer, TEST_PROMPTS)
    results["fft"]["responses"] = fft_after_responses
    
    # FFT 모델 크기
    fft_save_path = "./output/comparison_fft/full_model"
    fft_model.save_pretrained(fft_save_path)
    fft_size = sum(
        os.path.getsize(os.path.join(root, f))
        for root, dirs, files in os.walk(fft_save_path)
        for f in files
    )
    results["fft"]["save_size"] = fft_size
    print(f"📌 FFT 모델 크기: {fft_size/1024/1024:.0f} MB")
    
    del fft_model, fft_trainer
    gc.collect()
    torch.cuda.empty_cache()
else:
    results["fft"]["responses"] = ["(OOM - 학습 실패)"] * len(TEST_PROMPTS)
    results["fft"]["save_size"] = 3_000_000_000  # 추정값 ~3GB
    print("⚠️ FFT 학습이 실패하여 응답을 생성할 수 없습니다.")

print("✅ FFT 메모리 해제 완료")

## 4️⃣ 추론 품질 비교 (동일 프롬프트)

In [ ]:
print("="*70)
print("📊 추론 품질 비교: Before vs LoRA vs FFT")
print("="*70)

for i, prompt in enumerate(TEST_PROMPTS):
    print(f"\n{'='*70}")
    print(f"🔹 질문 {i+1}: {prompt[:70]}")
    print(f"\n  [Before]  {before_responses[i][:200]}")
    print(f"\n  [LoRA]    {results['lora']['responses'][i][:200]}")
    print(f"\n  [FFT]     {results['fft']['responses'][i][:200]}")

print(f"\n{'='*70}")

## 5️⃣ 학습 곡선(loss) 비교

In [ ]:
# 학습 로그에서 loss 추출
def extract_losses(log_history):
    """학습 로그에서 loss 값만 추출"""
    losses = []
    steps = []
    for entry in log_history:
        if "loss" in entry:
            losses.append(entry["loss"])
            steps.append(entry.get("step", len(losses)))
    return steps, losses

lora_steps, lora_losses = extract_losses(results["lora"].get("log_history", []))

print("📊 LoRA 학습 곡선 (텍스트)")
print("="*50)
if lora_losses:
    for step, loss in zip(lora_steps, lora_losses):
        bar = "█" * int(loss * 10)
        print(f"  Step {step:3d}: {loss:.4f} {bar}")

if results["fft"]["loss"] != float('inf'):
    fft_steps, fft_losses = extract_losses(results["fft"].get("log_history", []))
    print(f"\n📊 FFT 학습 곡선 (텍스트)")
    print("="*50)
    if fft_losses:
        for step, loss in zip(fft_steps, fft_losses):
            bar = "█" * int(loss * 10)
            print(f"  Step {step:3d}: {loss:.4f} {bar}")
else:
    print("\n⚠️ FFT 학습이 실패하여 loss 곡선을 표시할 수 없습니다.")

In [ ]:
# matplotlib이 있으면 시각화
try:
    import matplotlib.pyplot as plt
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    
    if lora_losses:
        ax.plot(lora_steps, lora_losses, 'b-o', label='LoRA', markersize=3)
    
    if results["fft"]["loss"] != float('inf'):
        fft_steps, fft_losses = extract_losses(results["fft"].get("log_history", []))
        if fft_losses:
            ax.plot(fft_steps, fft_losses, 'r-o', label='FFT', markersize=3)
    
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.set_title('LoRA vs FFT 학습 곡선 비교')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("./output/loss_comparison.png", dpi=100)
    plt.show()
    print("✅ 그래프 저장: ./output/loss_comparison.png")
    
except ImportError:
    print("ℹ️ matplotlib이 없어 텍스트로만 표시합니다. (pip install matplotlib)")

## 6️⃣ 종합 비교표 (메모리, 시간, 성능, 저장 크기)

In [ ]:
print("="*75)
print("📊 LoRA vs FFT 종합 비교 결과")
print("="*75)

fft_status = "성공" if results["fft"]["loss"] != float('inf') else "실패 (OOM)"
fft_time_str = f"{results['fft']['time']:.1f}초" if results["fft"]["time"] != float('inf') else "N/A (OOM)"
fft_loss_str = f"{results['fft']['loss']:.4f}" if results["fft"]["loss"] != float('inf') else "N/A"

print(f"\n{'항목':<25} {'LoRA':<25} {'FFT':<25}")
print("-"*75)
print(f"{'학습 상태':<25} {'✅ 성공':<25} {fft_status:<25}")
print(f"{'학습 시간':<25} {f'{results["lora"]["time"]:.1f}초':<25} {fft_time_str:<25}")
print(f"{'Final Loss':<25} {f'{results["lora"]["loss"]:.4f}':<25} {fft_loss_str:<25}")
print(f"{'GPU 메모리':<25} {f'~{results["lora"]["memory"]:.1f}GB':<25} {f'~{results["fft"]["memory"]:.1f}GB':<25}")
print(f"{'저장 크기':<25} {f'{results["lora"]["save_size"]/1024/1024:.1f}MB':<25} {f'{results["fft"]["save_size"]/1024/1024:.0f}MB':<25}")
print(f"{'학습 파라미터':<25} {'~1%':<25} {'100%':<25}")
print(f"{'RTX 4060 적합도':<25} {'✅ 매우 적합':<25} {'⚠️ 빠듯/불가':<25}")
print("-"*75)

print(f"\n📌 결론:")
print(f"   RTX 4060(8GB) 환경에서는 LoRA가 안정적이고 실용적입니다.")
print(f"   FFT는 충분한 VRAM(24GB+)이 있을 때 고려하세요.")

In [ ]:
# 임시 파일 정리
import shutil
for path in ["./output/comparison_lora", "./output/comparison_fft"]:
    if os.path.exists(path):
        shutil.rmtree(path)
if os.path.exists("./output/loss_comparison.png"):
    os.remove("./output/loss_comparison.png")
print("✅ 임시 파일 정리 완료")

## 📝 정리 및 핵심 요약

### 이번 실습에서 배운 내용

- 🎯 **동일 데이터, 동일 조건**에서 LoRA와 FFT를 직접 비교했습니다
- 🎯 **LoRA는 메모리 효율적**이면서도 합리적인 성능을 달성합니다
- 🎯 **FFT는 더 많은 자원**이 필요하지만, 최고 성능을 낼 수 있습니다
- 🎯 **RTX 4060에서 FFT는 1.5B까지만** 시도할 수 있고, 빠듯합니다

### 실무 가이드

| GPU VRAM | 추천 방법 | 최대 모델 크기 |
|----------|----------|---------------|
| 8GB (RTX 4060) | QLoRA | 7B |
| 16GB (RTX 4080) | LoRA | 7B, QLoRA 13B |
| 24GB (RTX 4090) | LoRA/FFT | FFT 3B, LoRA 13B |
| 48GB+ (A6000, A100) | FFT | FFT 13B+ |

### 다음 단계

- ➡️ **Notebook 21**: Unsloth 기반 파인튜닝 - 2배 빠른 학습, 60% 메모리 절약!